# A1.2 — Pilot Food-101 trên GPU Colab

Chọn **Runtime → Change runtime type → GPU**, chỉnh hai đường dẫn Drive bên dưới và chạy từ trên xuống. Pilot chỉ dùng 5 ảnh train và 2 ảnh validation mỗi lớp; không nạp ảnh test, không tải lại dataset hoặc tạo lại manifest. Đây không phải tám thí nghiệm đầy đủ.

In [ ]:
import platform
import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError('Cần bật GPU CUDA trong Colab trước khi chạy pilot')
runtime_info = {
    'python': platform.python_version(),
    'pytorch': torch.__version__,
    'torchvision': torchvision.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
}
print(runtime_info)

## Gắn Drive và kiểm tra source

Đường dẫn `DATASET_ROOT` phải chứa `meta/classes.txt`, `meta/train.json` và `images/`; `MANIFEST` là manifest hiện có trên Drive. Không tự tạo các đường dẫn dữ liệu bị thiếu.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DRIVE = Path('/content/drive/MyDrive/CO5085/A1-2')
DATASET_ROOT = PROJECT_DRIVE / 'data' / 'food-101'  # Chỉnh theo vị trí dataset đã giải nén.
MANIFEST = PROJECT_DRIVE / 'manifests' / 'food101_split_seed42.csv'
OUTPUT_BASE = PROJECT_DRIVE / 'outputs' / 'pilot'
for path in (DATASET_ROOT / 'meta' / 'classes.txt', DATASET_ROOT / 'meta' / 'train.json', MANIFEST):
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu tệp: {path}')
if not (DATASET_ROOT / 'images').is_dir():
    raise FileNotFoundError(f'Thiếu thư mục ảnh: {DATASET_ROOT / "images"}')
print('Dataset:', DATASET_ROOT, '\nManifest:', MANIFEST)

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/hoctran19-cyber/deep-learning-computer-vision-co5085.git'
REPO_DIR = Path('/content/deep-learning-computer-vision-co5085')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', 'assignment_A1', '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / '.git').exists():
        raise RuntimeError(f'{REPO_DIR} không phải Git repository')
    branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    remote = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    status = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True)
    if branch != 'assignment_A1' or remote.rstrip('/').removesuffix('.git') != REPO_URL.removesuffix('.git') or status:
        raise RuntimeError('Repository phải sạch, đúng origin và đang ở nhánh assignment_A1')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'assignment_A1'], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', commit)
sys.path.insert(0, str(REPO_DIR / 'A1-2'))

## Kiểm tra manifest và tạo subset

`Food101Dataset` kiểm tra cấu trúc manifest và sự tồn tại của ảnh train/validation; pilot đối chiếu train + validation với `train.json`, rồi chọn ngẫu nhiên theo lớp với seed 42. Không khởi tạo dataset test.

In [ ]:
import copy
import hashlib
import json
import random
import time
from collections import Counter
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import yaml

from src.dataset import create_dataloader
from src.fit import fit
from src.models import build_model, configure_finetuning, count_parameters
from src.optim import build_optimizer, build_scheduler
from src.pilot import prepare_pilot_datasets
from src.transforms import build_eval_transform, build_train_transform

with (REPO_DIR / 'A1-2' / 'configs' / 'config.yaml').open(encoding='utf-8') as file:
    base_config = yaml.safe_load(file)
data_config = base_config['data']
train_transform = build_train_transform(
    data_config['resize_short_edge'], data_config['image_size'],
    data_config['horizontal_flip_probability'])
eval_transform = build_eval_transform(data_config['resize_short_edge'], data_config['image_size'])
train_subset, validation_subset = prepare_pilot_datasets(
    DATASET_ROOT, MANIFEST, train_transform, eval_transform, seed=42)
for subset, per_class in ((train_subset, 5), (validation_subset, 2)):
    counts = Counter(subset.dataset.samples[index]['class_name'] for index in subset.indices)
    if len(counts) != 101 or any(count != per_class for count in counts.values()):
        raise ValueError(f'Class distribution invalid: {subset.split}')
print('Train:', len(train_subset), 'Validation:', len(validation_subset))
manifest_hash = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
run_dir = OUTPUT_BASE / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8])
run_dir.mkdir(parents=True, exist_ok=False)
print('Output:', run_dir)

## Chạy hai pilot from-scratch

Mỗi kiến trúc được reset seed 42 và có checkpoint/log riêng. Thời gian là wall-clock của `fit()`, gồm train, validation và ghi artifact, không phải train-only time theo A1-1.

In [ ]:
results = []
for architecture in ('resnet50', 'vit_b16'):
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    config = copy.deepcopy(base_config)
    config['training'].update(batch_size=8, max_epochs=2, regime='from_scratch', num_workers=0)
    config['training']['scheduler']['cosine_annealing']['T_max'] = 2
    train_loader = create_dataloader(train_subset, batch_size=8, num_workers=0, pin_memory=True)
    validation_loader = create_dataloader(validation_subset, batch_size=8, num_workers=0, pin_memory=True)
    model = build_model(architecture, num_classes=101, pretrained=False).to('cuda')
    configure_finetuning(model, 'from_scratch')
    total_params, trainable_params = count_parameters(model)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config)
    checkpoint_path = run_dir / f'{architecture}_best.pt'
    log_path = run_dir / f'{architecture}_epochs.jsonl'
    torch.cuda.synchronize()
    start = time.perf_counter()
    result = fit(model, train_loader, validation_loader, torch.nn.CrossEntropyLoss(), optimizer,
                 torch.device('cuda'), config, checkpoint_path, log_path, scheduler)
    torch.cuda.synchronize()
    wall_seconds = time.perf_counter() - start
    with log_path.open(encoding='utf-8') as file:
        epochs = [json.loads(line) for line in file]
    summary = {
        'architecture': architecture, 'regime': 'from_scratch', 'pretrained': False,
        'seed': 42, 'train_samples': len(train_subset), 'validation_samples': len(validation_subset),
        'trainable_parameters': trainable_params, 'total_parameters': total_params,
        'runtime': runtime_info, 'source_commit': commit, 'manifest_sha256': manifest_hash,
        'fit_wall_seconds_including_validation_and_io': wall_seconds, 'batch_size': 8,
        'config': config, 'epoch_metrics': epochs,
        'best_epoch': result['best_epoch'], 'validation_macro_f1': result['best_macro_f1'],
        'validation_accuracy': result['best_accuracy'], 'checkpoint': str(checkpoint_path),
        'log': str(log_path),
    }
    with (run_dir / f'{architecture}_summary.json').open('w', encoding='utf-8') as file:
        json.dump(summary, file, indent=2, allow_nan=False)
    results.append(summary)
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print(f"{'Model':<12} {'Best epoch':>10} {'Macro-F1':>10} {'Accuracy':>10} {'Fit seconds':>12}")
for row in results:
    print(f"{row['architecture']:<12} {row['best_epoch']:>10} {row['validation_macro_f1']:>10.4f} {row['validation_accuracy']:>10.4f} {row['fit_wall_seconds_including_validation_and_io']:>12.1f}")
print('Artifacts:', run_dir)